# SHIPP — Run the RAG stage end to end

```text
50 listing files → 51 vision → 52 listing content → 53 search docs → 54 AI Search index → 55 eval → 56 semantic scores
```
Run `pipeline/90_run_pipeline` first (Silver listings must be current). Appends one row per stage to
`pipeline_run_log`. Set `RUN_VISION = False` to reuse existing vision results; `RUN_EVAL = False` to skip 55.
**Clear state and outputs → Run all.** Stops at the first failed gate.

In [0]:
%run ../common/shipp_silver_lib

In [0]:
from config import tables, settings
from rag import index as rag_index
from databricks.sdk import WorkspaceClient

print("=== RUNTIME CODE SOURCES ===")
print("tables.py:", tables.__file__)
print("settings.py:", settings.__file__)
print("rag/index.py:", rag_index.__file__)

print("\n=== EXPECTED RESOURCES ===")
print("Vector Search endpoint:", settings.VS_ENDPOINT_NAME)
print("AI Search index:", tables.VS_INDEX_NAME)
print("Embedding endpoint:", settings.EMBEDDING_ENDPOINT)
print("Gold source table:", tables.GOLD_LISTING_SEARCH_DOCS)

w = WorkspaceClient()

checks = [
    (
        "Vector Search endpoint",
        lambda: w.vector_search_endpoints.get_endpoint(
            endpoint_name=settings.VS_ENDPOINT_NAME
        ),
    ),
    (
        "AI Search index",
        lambda: w.vector_search_indexes.get_index(
            index_name=tables.VS_INDEX_NAME
        ),
    ),
    (
        "Embedding endpoint",
        lambda: w.serving_endpoints.get(
            settings.EMBEDDING_ENDPOINT
        ),
    ),
]

for name, check in checks:
    try:
        resource = check()
        print(f"\n{name}: FOUND")

        if name == "AI Search index":
            status = resource.status
            print("Ready:", status.ready if status else None)
            print(
                "Message:",
                status.message if status else None
            )
            print(
                "Indexed rows:",
                status.indexed_row_count if status else None
            )

    except Exception as exc:
        print(f"\n{name}: ERROR")
        print("Exception:", type(exc).__name__)
        print("Message:", str(exc)[:1000])

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

canonical = (
    "/Users/raneem99hamarneh@yahoo.com/"
    "Shipp/shipp-marketplaceV4/notebooks/rag/"
    "54_ai_search_index"
)

obj = w.workspace.get_status(path=canonical)

print("=== CANONICAL NOTEBOOK ===")
print("Path:", obj.path)
print("Type:", obj.object_type)
print("ACCESS: PASS")

In [0]:
result = dbutils.notebook.run(
    "../rag/54_ai_search_index",
    3600
)

print("=== STAGE 54 RESULT ===")
print(result)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

folder = (
    "/Users/raneem99hamarneh@yahoo.com/"
    "Shipp/shipp-marketplaceV4/notebooks/rag"
)

print("=== ACTUAL WORKSPACE NOTEBOOKS ===")

try:
    objects = list(w.workspace.list(path=folder))

    for obj in objects:
        if "54_ai_search_index" in obj.path:
            print("Path:", obj.path)
            print("Type:", obj.object_type)
            print("---")

except Exception as exc:
    print("WORKSPACE ACCESS ERROR")
    print(type(exc).__name__, str(exc)[:500])

print("\n=== EXPECTED STAGE 54 PATH ===")

for stage, path in STAGES:
    if stage == "54_ai_search_index":
        print("Configured relative path:", path)

canonical = folder + "/54_ai_search_index"

try:
    obj = w.workspace.get_status(path=canonical)
    print("Canonical notebook: ACCESSIBLE")
    print("Actual path:", obj.path)
    print("Type:", obj.object_type)
except Exception as exc:
    print("Canonical notebook: NOT ACCESSIBLE")
    print(type(exc).__name__, str(exc)[:500])

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import RunType

w = WorkspaceClient()

print("=== RECENT NOTEBOOK WORKFLOW RUNS ===")

runs = list(
    w.jobs.list_runs(
        run_type=RunType.WORKFLOW_RUN,
        limit=20
    )
)

for run in runs:
    task = getattr(run, "task", None)
    notebook_task = getattr(task, "notebook_task", None)
    path = getattr(notebook_task, "notebook_path", None)

    state = getattr(run, "state", None)

    print(
        "Run ID:", run.run_id,
        "| Name:", run.run_name,
        "| Path:", path,
        "| State:", getattr(state, "result_state", None)
    )
    print("URL:", run.run_page_url)
    print("---")

In [0]:
import uuid
from datetime import datetime, timezone

RUN_VISION = True
RUN_EVAL = True
STAGES = [
    ("50_silver_listing_files",   "../rag/50_silver_listing_files"),
    ("51_vision_enrichment",      "../rag/51_vision_enrichment"),
    ("52_silver_listing_content", "../rag/52_silver_listing_content"),
    ("53_gold_search_docs",       "../rag/53_gold_listing_search_docs"),
    ("54_ai_search_index",        "../rag/54_ai_search_index"),
    ("55_rag_eval",               "../rag/55_rag_retrieval_eval"),
    ("56_semantic_scores",        "../rag/56_silver_semantic_scores"),
]
if not RUN_VISION:
    STAGES = [s for s in STAGES if s[0] != "51_vision_enrichment"]
if not RUN_EVAL:
    STAGES = [s for s in STAGES if s[0] != "55_rag_eval"]

LOG_SCHEMA = ("run_id string, stage string, started_at timestamp, ended_at timestamp, "
              "seconds double, status string, summary string")
spark.sql(f"CREATE TABLE IF NOT EXISTS {PIPELINE_RUN_LOG} ({LOG_SCHEMA}) USING DELTA")
RUN_ID = str(uuid.uuid4())
print("RUN_ID =", RUN_ID)

In [0]:
log_rows, run_started = [], time.time()
try:
    for stage, path in STAGES:
        started, t0 = datetime.now(timezone.utc).replace(tzinfo=None), time.time()
        print(f"▶ {stage} ...")
        try:
            result, status = dbutils.notebook.run(path, 3600), "PASS"
        except Exception as e:
            result, status = f"{type(e).__name__}: {str(e)[:1000]}", "FAILED"
        seconds = round(time.time() - t0, 1)
        log_rows.append((RUN_ID, stage, started, datetime.now(timezone.utc).replace(tzinfo=None), seconds, status, result))
        print(f"  {status} in {seconds}s — {result}")
        if status == "FAILED":
            raise AssertionError(f"RAG pipeline stopped at {stage}. Open that notebook and run it to see the failing gate.")
finally:
    if log_rows:
        spark.createDataFrame(log_rows, LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(PIPELINE_RUN_LOG)
    print(f"Total RAG pipeline time: {time.time() - run_started:.1f}s")

print("PASS — RAG pipeline completed.")
display(spark.table(PIPELINE_RUN_LOG).filter(F.col("run_id") == RUN_ID).orderBy("started_at"))

In [0]:
spark.sql("""
    SELECT
        started_at,
        run_id,
        stage,
        status,
        seconds,
        summary
    FROM bootcamp_students.shipp_gold.pipeline_run_log
    WHERE stage = '54_ai_search_index'
    ORDER BY started_at DESC
    LIMIT 5
""").show(truncate=False)


